# Benchmark AVEX + Random Forest

Benchmark de modeles AVEX pre-entraines pour extraire des embeddings audio, puis comparer leurs performances en classification Random Forest (GroupKFold).

Logique alignee sur vos notebooks Dasheng: extraction label/id depuis filename, grouped labels CSV, GroupKFold par enregistrement source, macro-F1 et balanced accuracy.

In [1]:
import sys
import os
import re
import time
import json
import zipfile
import tempfile
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torchaudio

from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score, f1_score

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/audio_simon_moutier'
else:
    PROJECT_PATH = 'C:/Users/moutier/Desktop/CNRS/CNRS'

# =========================================================
# INSTALL DEPENDENCIES
# =========================================================

if IN_COLAB:
    print('Installing dependencies...')

    # IMPORTANT: version compatible avec les modèles EAT
    get_ipython().system('pip install -q transformers==4.36.2')

    # AVEX
    get_ipython().system(
        'pip install -q git+https://github.com/earthspecies/avex.git '
        'scikit-learn pandas numpy pyarrow torchaudio tqdm'
    )

else:
    print(
        'If needed:\n'
        'pip install transformers==4.36.2\n'
        'pip install git+https://github.com/earthspecies/avex.git '
        'scikit-learn pandas numpy pyarrow torchaudio tqdm'
    )

# =========================================================
# IMPORTS AFTER INSTALL
# =========================================================

import transformers
print("Transformers version:", transformers.__version__)

from avex import list_models, get_model_spec, load_model

print(f'Project path: {PROJECT_PATH}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installing dependencies...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Transformers version: 4.36.2
Project path: /content/drive/MyDrive/audio_simon_moutier
Device: cuda


In [2]:
def _to_2d_tensor(output):
    if isinstance(output, dict):
        output = output.get('embeddings', next(iter(output.values())))
    if isinstance(output, (list, tuple)):
        return torch.cat([_to_2d_tensor(x) for x in output], dim=1)
    if not torch.is_tensor(output):
        output = torch.tensor(output)
    if output.dim() == 1:
        return output.unsqueeze(0)
    if output.dim() == 2:
        return output
    if output.dim() == 3:
      # CLS token si présent
      if output.shape[1] > 1:
          return output[:, 0, :]
      return output.mean(dim=1)
    return output.flatten(start_dim=1)


def extract_avex_embedding(model, wav_1d, sample_rate, target_sr, device):
    if sample_rate != target_sr:
        wav_1d = torchaudio.functional.resample(
            wav_1d.unsqueeze(0),
            orig_freq=sample_rate,
            new_freq=target_sr,
        ).squeeze(0)
    audio = wav_1d.unsqueeze(0).float().to(device)
    with torch.no_grad():
        try:
            out = model(audio)
        except TypeError:
            out = model(audio, padding_mask=None)
    return _to_2d_tensor(out)[0].detach().cpu().numpy()


def parse_metadata(df):
    out = df.copy()
    out['label'] = out['filename'].str.extract(r'(.*)(?=_HiP)')
    out['label'] = out['label'].astype('category')
    out['id'] = out['filename'].str.extract(r'(HiP[^_]+)')
    out['specie'] = np.where(
        out['id'].isin(['HiPsh441', 'HiPsh435']),
        'hyaena',
        np.where(out['id'].isin(['HiP616', 'HiP320', 'HiP633']), 'lion', 'unknown')
    )
    return out


def filter_rare_classes(df):
    counts = df['label'].value_counts()
    valid = counts[counts >= RARE_CLASS_THRESHOLD].index.tolist() + EXCEPTIONS
    return df[df['label'].isin(valid)].copy()


def apply_grouped_labels(df):
    labels_df = pd.read_csv(GROUPED_LABELS_CSV)
    group_map = {
        1: 'background',
        2: 'crunch',
        3: 'roar',
        4: 'whoop',
        5: 'prey_scream',
        6: 'h_noise',
        7: 'l_noise',
        8: 'whoop_o',
        9: 'roar_o',
    }
    labels_df['group_name'] = labels_df['group'].map(group_map).fillna('unclassified')
    if GROUP_NOISE:
        labels_df['group_name'] = labels_df['group_name'].replace({'h_noise': 'noise', 'l_noise': 'noise'})
    return df.merge(labels_df[['label', 'group_name']], on='label', how='left')


def build_sample_weights(y):
    class_counts = y.value_counts()
    if not USE_CUSTOM_WEIGHTS:
        w = y.map(lambda x: 1 / class_counts[x])
    else:
        priority = {
            'background': 1,
            'noise': 1,
            'crunch': 1,
            'roar': 1.5,
            'roar_o': 1.5,
            'whoop': 2,
            'whoop_o': 2,
        }
        w = y.map(lambda x: (1 / class_counts[x]) * priority.get(x, 1))
    return w / w.mean()


def train_evaluate_rf(df_features, output_dir):
    feature_cols = [c for c in df_features.columns if c.startswith('embedding_')]
    X = df_features[feature_cols]
    y = df_features['group_name']

    df_features = df_features.copy()
    df_features['original_file_id'] = df_features['filename'].str.extract(r'(HiP.+?)(?=_idx)')
    groups = df_features['original_file_id']

    cv_splits = 3 if REDUCE_CV_SPLITS else 5
    gkf = GroupKFold(n_splits=cv_splits)

    rf = RandomForestClassifier(
        n_estimators=200,
        max_features='sqrt',
        criterion='gini',
        min_samples_leaf=5,
        n_jobs=-1,
        class_weight=None,
        random_state=42,
    )

    weights = build_sample_weights(y)
    y_pred = np.empty(len(y), dtype=object)

    t_rf_start = time.time()
    for train_idx, test_idx in gkf.split(X, y, groups):
        rf.fit(X.iloc[train_idx], y.iloc[train_idx], sample_weight=weights.iloc[train_idx])
        y_pred[test_idx] = rf.predict(X.iloc[test_idx])

    macro_f1 = f1_score(y, y_pred, average='macro')
    balanced_acc = balanced_accuracy_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)

    # final fit on all data (for export)
    rf.fit(X, y, sample_weight=weights)
    t_rf_end = time.time()
    rf_training_s = t_rf_end - t_rf_start

    os.makedirs(output_dir, exist_ok=True)
    pd.DataFrame(cm).to_csv(os.path.join(output_dir, 'confusion_matrix.csv'), index=False)
    with open(os.path.join(output_dir, 'classification_report.txt'), 'w', encoding='utf-8') as f:
        f.write(classification_report(y, y_pred))
    joblib.dump(rf, os.path.join(output_dir, 'random_forest_model.pkl'))

    return {
        'macro_f1': macro_f1,
        'balanced_accuracy': balanced_acc,
        'n_samples': len(df_features),
        'n_features': len(feature_cols),
        'n_classes': y.nunique(),
        'class_distribution': y.value_counts().to_dict(),
        'rf_training_s': rf_training_s,
    }

print('Helpers ready')

Helpers ready


In [3]:
# Configuration
ZIP_PATH = os.path.join(PROJECT_PATH, 'Data/Audio/AudioRaw/3s_samples_subset.zip')
RESULTS_ROOT = os.path.join(PROJECT_PATH, 'Results/Random_Forest')
GROUPED_LABELS_CSV = os.path.join(RESULTS_ROOT, 'grouped_labels_V2.csv')
EMBEDDINGS_EXPORT_DIR = os.path.join(PROJECT_PATH, 'embeddings', 'avex')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Models to benchmark: edit this list directly to simplify selection.
MODELS_TO_BENCHMARK = [
    'esp_aves2_naturelm_audio_v1_beats',
    'esp_aves2_sl_beats_all',
    'esp_aves2_sl_beats_bio',
    'esp_aves2_eat_all',
    'esp_aves2_eat_bio'
]

MAX_FILES = None
USE_GROUPED_LABELS = True
GROUP_NOISE = True
RARE_CLASS_THRESHOLD = 30
REDUCE_CV_SPLITS = False
USE_CUSTOM_WEIGHTS = True

EXCEPTIONS = [
    'roar_period', 'wildebeest_scream', 'lion_roar', 'buffalo',
    'roar_o_period', 'lion_roar_period', 'whoop_o_period', 'scream_prey',
    'growl_thr', 'growl_thr_o', 'alarm_call_o', 'jap_o',
    'rumble_o', 'warthog', 'whine', 'scream',
]

os.makedirs(RESULTS_ROOT, exist_ok=True)
os.makedirs(EMBEDDINGS_EXPORT_DIR, exist_ok=True)

print('Models to benchmark:', MODELS_TO_BENCHMARK)
print(f'Device: {DEVICE}')

Models to benchmark: ['esp_aves2_naturelm_audio_v1_beats', 'esp_aves2_sl_beats_all', 'esp_aves2_sl_beats_bio', 'esp_aves2_eat_all', 'esp_aves2_eat_bio']
Device: cuda


In [4]:
# Data retrieval + shared preprocessing
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'ZIP not found: {ZIP_PATH}')
if USE_GROUPED_LABELS and not os.path.exists(GROUPED_LABELS_CSV):
    raise FileNotFoundError(f'Grouped labels CSV not found: {GROUPED_LABELS_CSV}')

EXTRACT_DIR = tempfile.mkdtemp(prefix='avex_extract_')
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

audio_files = sorted(Path(EXTRACT_DIR).rglob('*.wav'))
if MAX_FILES is not None:
    audio_files = audio_files[:MAX_FILES]

filenames_all = [p.name for p in audio_files]
benchmark_base_df = pd.DataFrame({'filename': filenames_all})
benchmark_base_df = parse_metadata(benchmark_base_df)

print('\nPre-extraction: total audio files=', len(benchmark_base_df))
print('Unique raw labels=', benchmark_base_df['label'].nunique())
print('Top raw label counts:\n', benchmark_base_df['label'].value_counts().head(20).to_string())

benchmark_base_df = filter_rare_classes(benchmark_base_df)
print('\nAfter filter_rare_classes: total=', len(benchmark_base_df), 'unique labels=', benchmark_base_df['label'].nunique())
print(benchmark_base_df['label'].value_counts().head(20).to_string())

if USE_GROUPED_LABELS:
    benchmark_base_df = apply_grouped_labels(benchmark_base_df)
    print('\nAfter apply_grouped_labels: n_groups=', benchmark_base_df['group_name'].nunique())
    print(benchmark_base_df['group_name'].value_counts().head(20).to_string())
else:
    benchmark_base_df['group_name'] = benchmark_base_df['label']
    print('\nGrouped labels disabled: using raw labels as group_name')
    print(benchmark_base_df['group_name'].value_counts().head(20).to_string())

benchmark_base_df['original_file_id'] = benchmark_base_df['filename'].str.extract(r'(HiP.+?)(?=_idx)')
benchmark_base_df = benchmark_base_df.dropna(subset=['group_name']).copy()

print('\nShared benchmark dataset ready:')
print('n_samples=', len(benchmark_base_df), 'n_classes=', benchmark_base_df['group_name'].nunique())


Pre-extraction: total audio files= 10325
Unique raw labels= 71
Top raw label counts:
 label
background    2393
crunch        1827
walk          1383
breath         489
run            442
vocalise       397
trot           345
breath_hi      304
roar_o         268
lick           234
bird           178
roar           177
wind           159
whoop_o        131
snore          122
vocalise_o     119
growl_res      109
scratch        103
meow            94
heartbeat       78

After filter_rare_classes: total= 9946 unique labels= 37
label
background    2393
crunch        1827
walk          1383
breath         489
run            442
vocalise       397
trot           345
breath_hi      304
roar_o         268
lick           234
bird           178
roar           177
wind           159
whoop_o        131
snore          122
vocalise_o     119
growl_res      109
scratch        103
meow            94
heartbeat       78

After apply_grouped_labels: n_groups= 7
group_name
noise         3865
background  

In [ ]:
# Benchmark loop
all_models = list(list_models().keys())
print("MODELS DISPONIBLES:")
for m in all_models:
    print(m)

available_model_names = list(list_models().keys())

def resolve_model(name):
    if name in available_model_names:
        return name
    # fallback: match partiel
    matches = [m for m in available_model_names if name in m]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"[WARN] Multiple matches for {name}: {matches}")
        return matches[0]
    print(f"[ERROR] Model not found: {name}")
    return None

selected_models = [resolve_model(m) for m in MODELS_TO_BENCHMARK]
selected_models = [m for m in selected_models if m is not None]
missing_models = [model_name for model_name in MODELS_TO_BENCHMARK if model_name not in available_model_names]

print('Selected models for benchmark:', selected_models)
if missing_models:
    print('Unavailable models skipped:', missing_models)

run_tag = datetime.now().strftime('%Y%m%d_%H%M%S')
benchmark_output_dir = os.path.join(RESULTS_ROOT, f'RF_AVEX_benchmark_{run_tag}')
os.makedirs(benchmark_output_dir, exist_ok=True)

all_results = []
start_global = time.time()

for model_name in selected_models:
    print(f'\n=== {model_name} ===')
    t0 = time.time()

    try:
        spec = get_model_spec(model_name)
        target_sr = spec.audio_config.sample_rate
        model = load_model(model_name, return_features_only=True, device=DEVICE)
        model.eval()
    except Exception as e:
        print(f'ERROR: Failed to load model {model_name}: {str(e)[:200]}')
        continue

    try:
        model_params_M = sum(p.numel() for p in model.parameters()) / 1e6
    except Exception:
        model_params_M = None

    filenames = []
    embs = []
    failed = 0

    t_embed_start = time.time()
    for audio_path in tqdm(audio_files, desc=f'Embeddings [{model_name}]'):
        try:
            wav, sr = torchaudio.load(str(audio_path))
            if wav.shape[0] > 1:
                wav = wav.mean(dim=0, keepdim=True)

            emb = extract_avex_embedding(model, wav.squeeze(0), sr, target_sr, DEVICE)
            embs.append(emb)
            filenames.append(audio_path.name)
        except Exception as e:
            failed += 1
            if failed <= 5:
                print(f'Error {audio_path.name}: {str(e)[:120]}')
    t_embed_end = time.time()
    embed_time_s = t_embed_end - t_embed_start

    if len(embs) == 0:
        print('No embeddings extracted, skip.')
        continue

    emb_array = np.stack(embs)
    emb_df = pd.DataFrame(emb_array, columns=[f'embedding_{i}' for i in range(emb_array.shape[1])])
    emb_df.insert(0, 'filename', filenames)

    print('\nEmbeddings dataframe shape:', emb_df.shape)
    print('First filenames sample:', filenames[:5])

    model_safe_name = re.sub(r'[^a-zA-Z0-9_\-]+', '_', model_name)
    emb_export_path = os.path.join(EMBEDDINGS_EXPORT_DIR, f'{model_safe_name}_{run_tag}.parquet')
    emb_df.to_parquet(emb_export_path, index=False)

    df = emb_df.merge(benchmark_base_df, on='filename', how='inner')
    df = df.dropna(subset=['group_name']).copy()
    print('Final dataset for RF: n_samples=', len(df), 'n_classes=', df['group_name'].nunique())

    model_output_dir = os.path.join(benchmark_output_dir, model_safe_name)
    metrics = train_evaluate_rf(df, model_output_dir)

    embeddings_size_mb = emb_array.nbytes / (1024 ** 2)

    all_results.append({
        'model_name': model_name,
        'embedding_dim': emb_array.shape[1],
        'model_params_M': model_params_M,
        'n_audio_files_input': len(audio_files),
        'n_embeddings_ok': len(embs),
        'n_failed': failed,
        'success_rate': len(embs) / len(audio_files) if len(audio_files) > 0 else 0,
        'macro_f1': metrics['macro_f1'],
        'balanced_accuracy': metrics['balanced_accuracy'],
        'embed_time_s': embed_time_s,
        'rf_training_s': metrics.get('rf_training_s', None),
        'embeddings_size_mb': embeddings_size_mb,
        'runtime_minutes': (time.time() - t0) / 60,
        'embeddings_parquet': emb_export_path,
        'rf_output_dir': model_output_dir,
    })

    print(f"Model summary: params_M={model_params_M}, embedding_dim={emb_array.shape[1]}, embed_time_s={embed_time_s:.2f}, rf_training_s={metrics.get('rf_training_s', None):.2f}, embeddings_size_mb={embeddings_size_mb:.2f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



Model Name                          Description                              Trained Classifier  
esp_aves2_eat_all                   eat_hf (pretrained backbone)             ❌ No                
esp_aves2_eat_bio                   eat_hf (pretrained backbone)             ❌ No                
esp_aves2_effnetb0_all              efficientnet (fine-tuned) - 12806 classes ✅ Yes (12806 classes)
esp_aves2_effnetb0_audioset         efficientnet (pretrained backbone)       ❌ No                
esp_aves2_effnetb0_bio              efficientnet (fine-tuned) - 12279 classes ✅ Yes (12279 classes)
esp_aves2_naturelm_audio_v1_beats   beats (pretrained backbone) - NatureLM   ❌ No                
esp_aves2_sl_beats_all              beats (fine-tuned) - 12806 classes - fine-tuned ✅ Yes (12806 classes)
esp_aves2_sl_beats_bio              beats (fine-tuned) - 12279 classes - fine-tuned ✅ Yes (12279 classes)
esp_aves2_sl_eat_all_ssl_all        eat_hf (fine-tuned) - 12806 classes      ✅ Yes (12806 classes

Embeddings [esp_aves2_naturelm_audio_v1_beats]: 100%|██████████| 10325/10325 [05:05<00:00, 33.84it/s]



Embeddings dataframe shape: (10325, 769)
First filenames sample: ['alarm_call_HiPsh435_audio_2022-03-06_23-59-48-060_file64_idx31681_1484.wav', 'alarm_call_HiPsh435_audio_2022-03-07_00-59-53-728_file65_idx31769_3405.wav', 'alarm_call_HiPsh435_audio_2022-03-07_00-59-53-728_file65_idx31775_2541.wav', 'alarm_call_HiPsh435_audio_2022-03-07_00-59-53-728_file65_idx31786_3067.wav', 'alarm_call_HiPsh435_audio_2022-03-07_00-59-53-728_file65_idx31819_2538.wav']
Final dataset for RF: n_samples= 9946 n_classes= 7
Model summary: params_M=90.717055, embedding_dim=768, embed_time_s=305.12, rf_training_s=284.61, embeddings_size_mb=30.25

=== esp_aves2_sl_beats_all ===


(…)s-all/esp-aves2-sl-beats-all.safetensors:   0%|          | 0.00/401M [00:00<?, ?B/s]

Embeddings [esp_aves2_sl_beats_all]: 100%|██████████| 10325/10325 [04:52<00:00, 35.33it/s]



Embeddings dataframe shape: (10325, 769)
First filenames sample: ['alarm_call_HiPsh435_audio_2022-03-06_23-59-48-060_file64_idx31681_1484.wav', 'alarm_call_HiPsh435_audio_2022-03-07_00-59-53-728_file65_idx31769_3405.wav', 'alarm_call_HiPsh435_audio_2022-03-07_00-59-53-728_file65_idx31775_2541.wav', 'alarm_call_HiPsh435_audio_2022-03-07_00-59-53-728_file65_idx31786_3067.wav', 'alarm_call_HiPsh435_audio_2022-03-07_00-59-53-728_file65_idx31819_2538.wav']
Final dataset for RF: n_samples= 9946 n_classes= 7
Model summary: params_M=90.717055, embedding_dim=768, embed_time_s=292.23, rf_training_s=293.13, embeddings_size_mb=30.25

=== esp_aves2_sl_beats_bio ===


(…)s-bio/esp-aves2-sl-beats-bio.safetensors:   0%|          | 0.00/399M [00:00<?, ?B/s]

Embeddings [esp_aves2_sl_beats_bio]:  40%|████      | 4161/10325 [01:57<03:17, 31.21it/s]

In [ ]:
# Summary and exports
if len(all_results) == 0:
    raise RuntimeError('Benchmark finished with no valid model result.')

summary_df = pd.DataFrame(all_results).sort_values('macro_f1', ascending=False).reset_index(drop=True)
summary_csv = os.path.join(benchmark_output_dir, 'benchmark_summary.csv')
summary_txt = os.path.join(benchmark_output_dir, 'benchmark_summary.txt')
summary_df.to_csv(summary_csv, index=False)

best_row = summary_df.iloc[0]
with open(summary_txt, 'w', encoding='utf-8') as f:
    f.write('============================================================\n')
    f.write(' AVEX PRETRAINED MODELS BENCHMARK (RANDOM FOREST)\n')
    f.write('============================================================\n\n')
    f.write(f'Date: {datetime.now()}\n')
    f.write(f'Total runtime: {(time.time() - start_global) / 60:.2f} minutes\n\n')
    f.write(summary_df[['model_name', 'macro_f1', 'balanced_accuracy', 'success_rate']].to_string(index=False))
    f.write('\n\n')
    f.write(
        f'Best model: {best_row["model_name"]} | '
        f'Macro-F1={best_row["macro_f1"]:.4f} | '
        f'BalancedAcc={best_row["balanced_accuracy"]:.4f}\n'
    )

print('Output folder:', benchmark_output_dir)
print('Summary CSV:', summary_csv)
print('Summary TXT:', summary_txt)
print(summary_df[['model_name', 'macro_f1', 'balanced_accuracy', 'embed_time_s', 'rf_training_s', 'embeddings_size_mb']].head(50).to_string(index=False))